<a href="https://colab.research.google.com/github/cuiandrew08-lab/LiDARFusionLearning/blob/main/BaselineTraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --force-reinstall numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 111.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-hea

In [1]:
import os
import numpy as np

from google.colab import drive
drive.mount("/content/drive", force_remount = False)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from tqdm.notebook import tqdm

import random

from torch.utils.data import Dataset, DataLoader

#import tensorflow as tf

TORCH_version = torch.__version__.split('+')[0]
#CUDA_version = torch.version.cuda.replace('.', '')

#!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{TORCH_version}+cu{CUDA_version}.html

#import torch_sparse

from scipy.ndimage import maximum_filter

import sys

#!pip install import-ipynb
#import import_ipynb

#!pip install open3d plotly
#import open3d as o3d

import matplotlib.pyplot as plt


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!npx degit google-research-datasets/Objectron/objectron objectron

!pip install --force-reinstall opencv-python-headless==4.9.0.80 &> /dev/null
!pip install nuscenes-devkit &> /dev/null

⠙⠹⠸⠼⠴⠦⠧⠇Need to install the following packages:
degit@3.8.0
Ok to proceed? (y) y

⠙⠹⠸> cloned google-research-datasets/Objectron#HEAD to objectron
⠙npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice
⠙

In [2]:
sys.path.insert(0, '/content')

from objectron.dataset.iou import IoU as IoU3d
from objectron.dataset.box import Box as BoxIoU

from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import LidarPointCloud, Box
from nuscenes.eval.detection.utils import category_to_detection_name
from nuscenes.utils.geometry_utils import points_in_box

nusc_root = "/content/drive/MyDrive/LiDARFusion/nuscenes/datanuscenes"

nusc = NuScenes(version='v1.0-mini', dataroot=nusc_root, verbose=True)

Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 0.448 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.


In [3]:
from pyquaternion import Quaternion

sys.path.insert(0, '/content/drive/MyDrive/LiDARFusion')

sys.path.append('/content/objectron/')

from voxel_pointnet2 import PointNetSetAbstraction

from centerpoint import CenterPoint

from lidar_baseline import PillarsEncoder, get_ij, get_point_pillar

import lidartrainlibrary as ltb

#import CenterPointHead

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pickle

dict_file_path = '/content/drive/MyDrive/LiDARFusion/gt_dict.pkl'

with open(dict_file_path, 'rb') as f:
    gt_paste = pickle.load(f)


In [5]:
class_names = ["barrier", "bicycle", "bus", "car", "construction_vehicle", "motorcycle", "pedestrian", "traffic_cone", "trailer", "truck"]

def check_bev_iou(new_box, existing):
  new_box_corners = new_box.corners()[0:2, [0,1,4,5]]

  for box in existing:
    corners = box.corners()[0:2,[0,1,4,5]]

    corners_x , corners_y = corners

    for coords in new_box_corners.T:
      x,y = coords

      if corners_x[corners_x<x].size > 0 and corners_x[corners_x>x].size > 0:
        if corners_y[corners_y<y].size >0 and corners_y[corners_y>y].size >0:

          return True

  return False

def sample_valid_placement(gt_boxes):
  x_min, y_min, x_max, y_max, z_min, z_max = ltb.get_boxes_max_min(gt_boxes)

  x_cand = np.random.uniform(x_min, x_max)

  y_cand = np.random.uniform(y_min, y_max)

  z_cand = np.random.uniform(z_min, z_max)

  return np.array([x_cand, y_cand, z_cand])

def gt_sampling(cloud, gt_boxes, gt_names, gt_dict, sample_groups): #sample_groups is dict of how many of each class to sample

  for cls, num_sample in sample_groups.items():
    candidates = random.sample(gt_dict[cls], num_sample)

    for cand in candidates:
      new_center = sample_valid_placement(gt_boxes)
      new_box = cand["box"]
      new_box.translate(new_center)

      detection_name = ltb.category_to_detection_name(new_box.name)
      new_box.name = detection_name

      label = ltb.category_to_label(new_box.name)
      new_box.label = label

      if check_bev_iou(new_box, gt_boxes):
        continue

      new_points = cand['points'].copy()
      new_points[:, :3] += new_center

      points = np.concatenate([cloud, new_points], axis=0)
      gt_boxes.append(new_box)
      gt_names.append(cls)

  return points, gt_boxes, gt_names

In [6]:
test_scene = nusc.scene[0]

token_0 = test_scene["first_sample_token"]

cloud_sample = nusc.get("sample", token_0)

boxes = nusc.get_boxes(cloud_sample["data"]["LIDAR_TOP"])

points = ltb.load_sweeps(nusc, cloud_sample)

gt_boxes = ltb.process_boxes(nusc, cloud_sample, boxes, points)

gt_names = ltb.get_box_names(gt_boxes)

sample_groups = {"car": 2, "pedestrian": 1, "construction_vehicle": 4}

samples = ltb.get_samples([0,1,2], nusc)

In [7]:
class LidarDetectionDataset(Dataset): #contains only the nusc samples from which points/boxes are pulled

  def __init__(self, samples, gt_paste_samplegroups, pc_range = [-51.2, -51.2, 51.2, 51.2]):

    self.samples = samples

    self.sample_groups = gt_paste_samplegroups
    self.pc_range = pc_range

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):

    sample_idx = self.samples[idx]

    points = ltb.load_sweeps(nusc, sample_idx)
    boxes = nusc.get_boxes(sample_idx["data"]["LIDAR_TOP"])
    boxes = ltb.process_boxes(nusc, sample_idx, boxes, points)
    names = ltb.get_box_names(boxes)

    points, boxes, names = gt_sampling(points, boxes, names, gt_paste, self.sample_groups)

    ltb.global_augmentation(points, boxes)

    points = ltb.cloud_filter_pcrange(points, pc_range = [self.pc_range[0], self.pc_range[2]])
    boxes, names = ltb.boxes_filter_pcrange(boxes, pc_range = [self.pc_range[0], self.pc_range[2]])

    pillars, coords, pillars_mask = ltb.get_pillars(points)

    heatmap, offset, size, z, rotation, index, mask, cat = ltb.target_generation(boxes)

    reg_targets = [offset, size, z, rotation]

    return {
        "pillars": pillars,
        "pillars_mask": pillars_mask,
        "coords": coords,
        "gt_boxes": boxes,
        "gt_labels": names,
        "heatmap": heatmap,
        "reg_target": reg_targets,
        "ind": index,
        "targets_mask": mask,
        "cat": cat
    }

In [8]:
def collate_fn(batch):
    voxels = torch.cat([b['pillars'] for b in batch], dim=0)
    num_points = torch.cat([b['pillars_mask'] for b in batch], dim=0)

    coords = []
    for i, b in enumerate(batch):
        coord_pad = F.pad(b['coords'], (1, 0), mode='constant', value=i)
        coords.append(coord_pad)
    coords = torch.cat(coords, dim=0)

    heatmap = torch.stack([b['heatmap'] for b in batch], dim=0)
    reg_target = [b['reg_target'] for b in batch]
    reg_mask = torch.stack([b['targets_mask'] for b in batch], dim=0)
    ind = torch.stack([b['ind'] for b in batch], dim=0)
    cat = torch.stack([b['cat'] for b in batch], dim=0)

    # gt_boxes/gt_labels stay as a list of variable-length tensors —
    # don't pad these for loss, only pull them out for stage-2 / eval
    gt_boxes = [b['gt_boxes'] for b in batch]
    gt_labels = [b['gt_labels'] for b in batch]

    return {
        'pillars': voxels, 'coords': coords, 'pillars_mask': num_points,
        'heatmap': heatmap, 'reg_target': reg_target, 'targets_mask': reg_mask,
        'ind': ind, 'cat': cat,
        'gt_boxes': gt_boxes, 'gt_labels': gt_labels,
    }

In [9]:
lidarset = LidarDetectionDataset(samples, sample_groups)

loader = DataLoader(lidarset, batch_size=2, shuffle=True,
                     collate_fn=collate_fn, num_workers=2, pin_memory=True)

In [10]:
class LIDARBaseline(nn.Module):

  def __init__(self, encoder, extracter, in_dims = [512,512], pc_range = [-51.2, 51.2]):
    super().__init__()

    self.encoder = encoder
    self.extracter = extracter
    self.in_dims = in_dims
    self.C = encoder.out_dim
    self.pc_range = pc_range
    self.pillar_size = (self.pc_range[1]-self.pc_range[0])/ self.in_dims[0]

  def get_batch_ids(self, coords):

    batch_ids = coords[:, 0:1]

    id_list = [0]
    for i in range(batch_ids.shape[0]-1):
      if batch_ids[i] != batch_ids[i+1]:
        id_list.append(i+1)

    id_list.append(batch_ids.shape[0]+1)

    return id_list

  def forward(self, pillars, coords, mask, train = False): #change to batched inputs

    boxes_list = []
    hm_list = []
    s1_centers_list = []
    refine_list = []
    I_list = []
    maps_list = []

    id_list = self.get_batch_ids(coords)

    for j in range(len(id_list)-1):
      start = id_list[j]
      stop = id_list[j+1]

      feature_map = torch.zeros((self.in_dims[0], self.in_dims[1], self.C),device = pillars.device)
      out = self.encoder(pillars[start:stop], mask[start:stop])

      for i in range(out.shape[0]):

        centroid = coords[start+i][1:3]
        x, y = torch.floor((centroid -self.pc_range[0]) / self.pillar_size)

        if x == self.in_dims[0]:
          x += -1

        if y == self.in_dims[1]:
          y += -1

        feature_map[int(x)][int(y)] = out[i]

      feature_map = feature_map.permute(2,0,1)

      boxes, heatmap, centers, dvec, I_0, regression_maps = self.extracter(feature_map)

      boxes_list.append(boxes)
      hm_list.append(heatmap)
      s1_centers_list.append(centers)
      refine_list.append(dvec)
      I_list.append(I_0)
      maps_list.append(regression_maps)

    heatmap_out = torch.stack(hm_list,dim=0)

    if train == True:

      return {"boxes": boxes_list, "heatmap": heatmap_out, "stage1_centers": s1_centers_list, "refine_input": refine_list, "I_input": I_list, "regression_maps": maps_list}

    return boxes

Encoder = PillarsEncoder(extra_features=2)
BoxHead = CenterPoint(512,512,256,10, K = 50) #[l, w, h, sin, cos, x,y,z ,k, I]
model = LIDARBaseline(Encoder, BoxHead)

In [14]:
def loss_hm(pred, target, alpha=2, beta=4):
    # pred, target: [B, C, H, W] — same shape, C = num classes
    pos_mask = (target == 1).float()
    neg_mask = (target < 1).float()

    pos_mask = pos_mask.to(pred.device)
    neg_mask = neg_mask.to(pred.device)

    pos_loss = torch.log(pred) * torch.pow(1 - pred, alpha) * pos_mask
    neg_loss = torch.log(1 - pred) * torch.pow(pred, alpha) * torch.pow(1 - target, beta) * neg_mask

    num_pos = pos_mask.sum()
    loss = -(pos_loss.sum() + neg_loss.sum())
    return loss / torch.clamp(num_pos, min=1)

class LidarLoss(nn.Module):

  def __init__(self, alpha = 2, beta = 4, lambda_reg = 0.25, hm_dims = 512):
    super(LidarLoss, self).__init__()
    self.alpha = alpha
    self.beta = beta
    self.lambda_reg = lambda_reg
    self.l1loss = nn.L1Loss()
    self.offsetl1 = nn.L1Loss(reduction="none")
    self.smoothl1 = nn.SmoothL1Loss()
    self.dims = hm_dims

  def match_candidates_to_gt(self, stage1_centers, gt_boxes, dist_thresh=2.0):
    matched_gt_idx = torch.full(((stage1_centers.shape[0]),), -1, dtype=torch.long)

    for i in range(stage1_centers.shape[0]):
      cand = stage1_centers[i]
      dists = torch.linalg.norm(gt_boxes[:, 5:7] - cand[5:7], dim=1)  # BEV center distance
      min_dist, min_idx = dists.min(dim=0)
      if min_dist < dist_thresh:
        matched_gt_idx[i] = min_idx
    return matched_gt_idx  # -1 = unmatched, no refinement target

  def generate_stage2_targets(self, stage1_centers, gt_boxes):
    matched_gt = self.match_candidates_to_gt(stage1_centers, gt_boxes)

    refine_targets = []
    Iou_targets = []

    for i in range(stage1_centers.shape[0]):

      g = matched_gt[i]

      delta_center = (gt_boxes[g, 5:8] - stage1_centers[i, 5:8])  # residual center offset
      delta_dim  = torch.log(torch.exp(gt_boxes[g, :3]) / torch.exp(stage1_centers[i, :3]))  # log-ratio, not log-absolute
      delta_rot  = torch.arcsin(gt_boxes[g, 3:4]) - torch.arcsin(stage1_centers[i, 3:4])  # residual yaw

      out_0 = torch.cat([delta_dim, delta_rot, delta_center])

      refine_targets.append(out_0)

      iou_0 = torch.tensor([ltb.IoU(stage1_centers[i], gt_boxes[g])], device = gt_boxes.device)
      Iou_targets.append(iou_0)

    delta_targets = torch.stack(refine_targets, dim = 0)
    I_targets = torch.stack(Iou_targets, dim=0)

    return delta_targets, I_targets

  def forward(self, heatmap_input, regression_maps, ind, mask, stage1_centers, refine_input, I_input, target_map, target_reg, gt_boxes): #batched inputs for all
  #change gt_boxes to work with collate function

    stage1_loss = 0
    stage2_loss = 0

    for j in range(heatmap_input.shape[0]):

      target_reg_0 = []
      reg_input_0 = []

      num_sample = (mask[j] == 1).nonzero()

      for m in range(num_sample.item()):
        x = int(ind[j][m] % self.dims)
        y = int(torch.floor(ind[j][m] / self.dims))

        offset = target_reg[j][0][x][y]
        size = target_reg[j][1][x][y]
        z = target_reg[j][2][x][y:y+1]
        rotation = target_reg[j][3][x][y]

        o_hat = regression_maps[j][0][x][y]
        size_hat = regression_maps[j][1][x][y]
        z_hat = regression_maps[j][2][x][y:y+1]
        rot_hat = regression_maps[j][3][x][y]

        target_reg_m = torch.cat([offset, size, z, rotation], dim=0)
        reg_input_m = torch.cat([o_hat, size_hat, z_hat, rot_hat], dim = 0)
        target_reg_0.append(target_reg_m)
        reg_input_0.append(reg_input_m)

      target_reg_0 = torch.stack(target_reg_0)
      reg_input_0 = torch.stack(reg_input_0)
      target_reg_0 = target_reg_0.to(reg_input_0.device)

      box_tensor = ltb.boxes_to_tensor(gt_boxes[j])
      box_tensor = box_tensor.to(stage1_centers[j].device)

      refine_mask = self.match_candidates_to_gt(stage1_centers[j], box_tensor)
      refine_mask = (refine_mask >= 0).float()
      refine_mask = refine_mask.to(heatmap_input.device)
      refine_target, I_targets = self.generate_stage2_targets(stage1_centers[j], box_tensor)

      target_map = target_map.to(heatmap_input.device)

      refine_in = refine_input[j]
      refine_in = refine_in.to(heatmap_input.device)
      I_in = I_input[j]
      I_in = I_in.to(heatmap_input.device)

      heatmap_loss = loss_hm(heatmap_input[j], target_map[j], alpha = self.alpha, beta = self.beta)

      reg_loss = self.l1loss(reg_input_0, target_reg_0)

      reg_loss = reg_loss * self.lambda_reg

      stage1_loss += heatmap_loss + reg_loss

      offset_loss = self.offsetl1(refine_in, refine_target)
      offset_loss = (offset_loss * refine_mask.unsqueeze(-1)).sum() / (refine_mask.sum() + 1e-4)

      confidence_loss = self.smoothl1(I_in, I_targets)

      stage2_loss += offset_loss + confidence_loss

    stage1_loss = stage1_loss / heatmap_input.shape[0]
    stage2_loss = stage2_loss / heatmap_input.shape[0]

    return stage1_loss + stage2_loss

loss_lidar = LidarLoss()

In [12]:
device = "cuda"
model = model.to(device)

In [15]:
for _ in tqdm(range(1)):

    for batch in loader:

      pillars, coords, mask = batch["pillars"], batch["coords"], batch["pillars_mask"]

      pillars, mask = pillars.to(device), mask.to(device) #clear these out of memory after loss/out is done
      out = model(pillars, coords, mask, train = True)

      loss = loss_lidar(out["heatmap"], out["regression_maps"], batch["ind"], batch["targets_mask"], out["stage1_centers"], out["refine_input"], out["I_input"], batch["heatmap"], batch["reg_target"], batch["gt_boxes"])



  0%|          | 0/1 [00:00<?, ?it/s]

torch.Size([3484, 100, 5])


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:1057: UserWarning: Using a target size (torch.Size([10, 1])) that is different to the input size (torch.Size([10])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.smooth_l1_loss(input, target, reduction=self.reduction, beta=self.beta)


tensor(3153.2777, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)
torch.Size([5168, 100, 5])


KeyboardInterrupt: 

In [13]:
batch = next(iter(loader))

In [25]:
out["refine_input"][0].device

device(type='cpu')

In [47]:
batch["reg_target"][1][2][1][511:512]

tensor([0.])

In [ ]:
test_data = lidarset.__getitem__(4)